# MCTS-200 vs Stockfish

Run the local McChess ResNet-B MCTS-200 bot against a local Stockfish UCI binary. This notebook is an external reference benchmark only. Do not use Stockfish moves, evaluations, game outcomes, or level settings as training labels.

The notebook alternates colors, shows the board while each game runs, and writes one JSON record per game plus a schedule summary under `runs/external_stockfish/`.

Before running, install Stockfish and either put `stockfish` on `PATH` or set `STOCKFISH_PATH` to the full executable path.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
for candidate in (project_root, *project_root.parents):
    if (candidate / "pyproject.toml").exists() and (candidate / "src" / "mcchess").exists():
        project_root = candidate
        break
else:
    raise RuntimeError("Could not find the McChess project root.")

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

In [ ]:
import asyncio
import csv
import datetime as dt
import html
import io
import json
import math
import shutil
import sys
import time
from typing import Any

import chess
import chess.engine
import chess.svg
from IPython.display import HTML, display

from mcchess.bots import MCTSBot

if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

In [ ]:
CHECKPOINT_PATH = project_root / "runs" / "lichess_2026_05_2000plus_resnet_b_epoch20_cached_batchmetrics" / "checkpoint.pt"
MCTS_SIMULATIONS = 200
C_PUCT = 1.5
INFERENCE_DEVICE = "auto"

STOCKFISH_PATH = r"C:\\Users\\mpjgl\\Desktop\\Stockfish\\stockfish-windows-x86-64-avx2\\stockfish\\stockfish-windows-x86-64-avx2.exe"

MAX_PLY = 180
SECONDS_BETWEEN_MOVES = 0.25

# Start with full Stockfish as a sanity check. Then run 20 paired games across
# Stockfish's UCI_Elo range 1600-2500. Every row resets Skill Level and
# UCI_LimitStrength so settings do not leak between levels.
STOCKFISH_SCHEDULE = [
    {"level": "full_stockfish_t1s_sanity", "games": 1, "options": {"Skill Level": 20, "UCI_LimitStrength": False}, "limit": {"time": 1.0}},
    {"level": "uci_elo_1600_t1s", "games": 2, "options": {"Skill Level": 20, "UCI_LimitStrength": True, "UCI_Elo": 1600}, "limit": {"time": 1.0}},
    {"level": "uci_elo_1700_t1s", "games": 2, "options": {"Skill Level": 20, "UCI_LimitStrength": True, "UCI_Elo": 1700}, "limit": {"time": 1.0}},
    {"level": "uci_elo_1800_t1s", "games": 2, "options": {"Skill Level": 20, "UCI_LimitStrength": True, "UCI_Elo": 1800}, "limit": {"time": 1.0}},
    {"level": "uci_elo_1900_t1s", "games": 2, "options": {"Skill Level": 20, "UCI_LimitStrength": True, "UCI_Elo": 1900}, "limit": {"time": 1.0}},
    {"level": "uci_elo_2000_t1s", "games": 2, "options": {"Skill Level": 20, "UCI_LimitStrength": True, "UCI_Elo": 2000}, "limit": {"time": 1.0}},
    {"level": "uci_elo_2100_t1s", "games": 2, "options": {"Skill Level": 20, "UCI_LimitStrength": True, "UCI_Elo": 2100}, "limit": {"time": 1.0}},
    {"level": "uci_elo_2200_t1s", "games": 2, "options": {"Skill Level": 20, "UCI_LimitStrength": True, "UCI_Elo": 2200}, "limit": {"time": 1.0}},
    {"level": "uci_elo_2300_t1s", "games": 2, "options": {"Skill Level": 20, "UCI_LimitStrength": True, "UCI_Elo": 2300}, "limit": {"time": 1.0}},
    {"level": "uci_elo_2400_t1s", "games": 2, "options": {"Skill Level": 20, "UCI_LimitStrength": True, "UCI_Elo": 2400}, "limit": {"time": 1.0}},
    {"level": "uci_elo_2500_t1s", "games": 2, "options": {"Skill Level": 20, "UCI_LimitStrength": True, "UCI_Elo": 2500}, "limit": {"time": 1.0}},
]

# Alternative skill-level schedule. Skill Level is not Elo-calibrated:
# STOCKFISH_SCHEDULE = [
#     {"level": "skill_0", "games": 2, "options": {"Skill Level": 0}, "limit": {"time": 0.05}},
#     {"level": "skill_5", "games": 2, "options": {"Skill Level": 5}, "limit": {"time": 0.05}},
#     {"level": "skill_10", "games": 2, "options": {"Skill Level": 10}, "limit": {"time": 0.05}},
# ]

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Missing checkpoint: {CHECKPOINT_PATH}")
if MCTS_SIMULATIONS != 200:
    raise ValueError("This notebook is intentionally fixed to MCTS-200")
if not STOCKFISH_PATH:
    raise FileNotFoundError("Set STOCKFISH_PATH or put stockfish on PATH")
if not Path(STOCKFISH_PATH).exists() and shutil.which(STOCKFISH_PATH) is None:
    raise FileNotFoundError(f"Stockfish binary not found: {STOCKFISH_PATH}")
if MAX_PLY <= 0:
    raise ValueError("MAX_PLY must be positive")
if not STOCKFISH_SCHEDULE:
    raise ValueError("STOCKFISH_SCHEDULE must contain at least one level")

In [ ]:
bot = MCTSBot.from_checkpoint(
    CHECKPOINT_PATH,
    device=INFERENCE_DEVICE,
    name=f"resnet_b_mcts_{MCTS_SIMULATIONS}",
    simulations=MCTS_SIMULATIONS,
    c_puct=C_PUCT,
)

metadata = bot.checkpoint.metadata
display(
    {
        "checkpoint": str(metadata.path),
        "epoch": metadata.epoch,
        "val_total_loss": metadata.metrics.get("val_total_loss"),
        "device": str(bot.device),
        "mcts_simulations": bot.config.simulations,
        "c_puct": bot.config.c_puct,
        "stockfish_path": STOCKFISH_PATH,
        "schedule": STOCKFISH_SCHEDULE,
    }
)

In [ ]:
BOARD_DISPLAY = None
TABLE_DISPLAY = None


def show_board(board: chess.Board, *, header: str = "", last_move: chess.Move | None = None) -> None:
    global BOARD_DISPLAY
    escaped_header = html.escape(header)
    moves = html.escape(" ".join(move.uci() for move in board.move_stack[-60:]))
    board_svg = chess.svg.board(board=board, lastmove=last_move, size=520)
    body = (
        "<div>"
        + (f"<b>{escaped_header}</b>" if header else "")
        + board_svg
        + (f"<pre>{moves}</pre>" if moves else "")
        + "</div>"
    )
    if BOARD_DISPLAY is None:
        BOARD_DISPLAY = display(HTML(body), display_id=True)
    else:
        BOARD_DISPLAY.update(HTML(body))


def configure_stockfish(engine: chess.engine.SimpleEngine, options: dict[str, Any]) -> dict[str, Any]:
    supported = {name: value for name, value in options.items() if name in engine.options}
    ignored = sorted(set(options) - set(supported))
    if supported:
        engine.configure(supported)
    if ignored:
        display({"ignored_stockfish_options": ignored})
    return supported


def stockfish_limit(limit_config: dict[str, Any]) -> chess.engine.Limit:
    allowed = {"time", "depth", "nodes", "mate"}
    unknown = sorted(set(limit_config) - allowed)
    if unknown:
        raise ValueError(f"Unsupported Stockfish limit keys: {unknown}")
    return chess.engine.Limit(**limit_config)


def score_from_result(result: str, mcchess_color: chess.Color) -> float:
    if result == "1/2-1/2":
        return 0.5
    if result == "1-0":
        return 1.0 if mcchess_color == chess.WHITE else 0.0
    if result == "0-1":
        return 1.0 if mcchess_color == chess.BLACK else 0.0
    raise ValueError(f"Unsupported result: {result}")


def winner_from_result(result: str) -> str:
    if result == "1-0":
        return "white"
    if result == "0-1":
        return "black"
    if result == "1/2-1/2":
        return "draw"
    return "unknown"


def color_name(color: chess.Color) -> str:
    return "white" if color == chess.WHITE else "black"


def game_table_html(games: list[dict[str, Any]]) -> str:
    rows = []
    for game in games:
        winner = html.escape(str(game["winner"]))
        winner_name = html.escape(str(game["winner_name"]))
        rows.append(
            "<tr>"
            f"<td>{game['game_index'] + 1}</td>"
            f"<td>{html.escape(str(game['level']))}</td>"
            f"<td>{html.escape(str(game['white']))}</td>"
            f"<td>{html.escape(str(game['black']))}</td>"
            f"<td>{html.escape(str(game['result']))}</td>"
            f"<td>{winner}</td>"
            f"<td>{winner_name}</td>"
            f"<td>{html.escape(str(game['mcchess_score']))}</td>"
            "</tr>"
        )
    if not rows:
        rows.append("<tr><td colspan='8'>No completed games yet.</td></tr>")
    return (
        "<style>.mcchess-results { border-collapse: collapse; margin-top: 12px; }"
        ".mcchess-results th, .mcchess-results td { border: 1px solid #ccc; padding: 4px 8px; text-align: left; }"
        ".mcchess-results th { background: #f3f3f3; }</style>"
        "<table class='mcchess-results'>"
        "<thead><tr>"
        "<th>Game</th><th>Stockfish level</th><th>White</th><th>Black</th><th>Result</th><th>Winner</th><th>Winner name</th><th>McChess score</th>"
        "</tr></thead>"
        "<tbody>"
        + "".join(rows)
        + "</tbody></table>"
    )

def show_game_table(games: list[dict[str, Any]]) -> None:
    global TABLE_DISPLAY
    content = HTML("<h3>Results</h3>" + game_table_html(games))
    if TABLE_DISPLAY is None:
        TABLE_DISPLAY = display(content, display_id=True)
    else:
        TABLE_DISPLAY.update(content)


def game_table_rows(games: list[dict[str, Any]]) -> list[dict[str, Any]]:
    return [
        {
            "game": game["game_index"] + 1,
            "stockfish_level": game["level"],
            "white": game["white"],
            "black": game["black"],
            "result": game["result"],
            "winner": game["winner"],
            "winner_name": game["winner_name"],
            "mcchess_score": game["mcchess_score"],
        }
        for game in games
    ]


def game_table_csv(games: list[dict[str, Any]]) -> str:
    output = io.StringIO()
    fieldnames = ["game", "stockfish_level", "white", "black", "result", "winner", "winner_name", "mcchess_score"]
    writer = csv.DictWriter(output, fieldnames=fieldnames, lineterminator="\n")
    writer.writeheader()
    writer.writerows(game_table_rows(games))
    return output.getvalue()


def write_game_table(output_dir: Path, games: list[dict[str, Any]]) -> dict[str, str]:
    html_path = output_dir / f"mcts_{MCTS_SIMULATIONS}_vs_stockfish_results_table.html"
    csv_path = output_dir / f"mcts_{MCTS_SIMULATIONS}_vs_stockfish_results_table.csv"
    html_path.write_text(game_table_html(games) + "\n", encoding="utf-8")
    csv_path.write_text(game_table_csv(games), encoding="utf-8")
    return {"html_table": str(html_path), "csv_table": str(csv_path)}


def expected_score(player_elo: int, opponent_elo: int) -> float:
    return 1.0 / (1.0 + 10.0 ** ((opponent_elo - player_elo) / 400.0))


def estimate_mcchess_elo(games: list[dict[str, Any]]) -> dict[str, Any]:
    observations = []
    for game in games:
        options = game.get("stockfish_options", {})
        if not options.get("UCI_LimitStrength") or "UCI_Elo" not in options:
            continue
        score = game.get("mcchess_score")
        if score is None:
            continue
        observations.append((int(options["UCI_Elo"]), float(score), game["game_index"] + 1))

    if not observations:
        return {
            "status": "unavailable",
            "reason": "No completed UCI_Elo games were available.",
        }

    def log_likelihood(player_elo: int) -> float:
        total = 0.0
        for opponent_elo, score, _ in observations:
            p = min(max(expected_score(player_elo, opponent_elo), 1e-9), 1.0 - 1e-9)
            total += score * math.log(p) + (1.0 - score) * math.log(1.0 - p)
        return total

    grid = list(range(800, 3201))
    likelihoods = {elo: log_likelihood(elo) for elo in grid}
    best_elo = max(grid, key=lambda elo: likelihoods[elo])
    best_ll = likelihoods[best_elo]
    # Rough 95% likelihood interval for one fitted parameter.
    interval_elos = [elo for elo in grid if best_ll - likelihoods[elo] <= 1.92]
    total_score = sum(score for _, score, _ in observations)
    return {
        "status": "estimated",
        "method": "one-parameter logistic maximum likelihood over Stockfish UCI_Elo games only",
        "estimated_elo": best_elo,
        "rough_95_interval": [min(interval_elos), max(interval_elos)] if interval_elos else None,
        "uci_elo_games": len(observations),
        "score": total_score,
        "score_rate": total_score / len(observations),
        "opponent_elos": [opponent_elo for opponent_elo, _, _ in observations],
        "included_game_numbers": [game_number for _, _, game_number in observations],
        "excluded": "Full Stockfish sanity games and non-UCI_Elo rows are excluded.",
        "caveat": "This is a rough Stockfish UCI_Elo benchmark estimate, not a Lichess or FIDE Elo claim.",
    }

In [ ]:
def play_stockfish_game(
    *,
    engine: chess.engine.SimpleEngine,
    level_name: str,
    game_index: int,
    mcchess_color: chess.Color,
    configured_options: dict[str, Any],
    limit_config: dict[str, Any],
) -> dict[str, Any]:
    board = chess.Board()
    moves = []
    started_at = dt.datetime.now(dt.timezone.utc).isoformat()
    limit = stockfish_limit(limit_config)
    illegal_move = None
    termination = "max_ply"

    while len(moves) < MAX_PLY:
        outcome = board.outcome(claim_draw=True)
        if outcome is not None:
            termination = outcome.termination.name.lower()
            break

        white_name = bot.name if mcchess_color == chess.WHITE else level_name
        black_name = level_name if mcchess_color == chess.WHITE else bot.name

        if board.turn == mcchess_color:
            move = bot.choose_move(board.copy(stack=True))
            actor = bot.name
            actor_color = color_name(board.turn)
        else:
            engine_result = engine.play(board, limit)
            if engine_result.move is None:
                raise RuntimeError("Stockfish returned no move")
            move = engine_result.move
            actor = level_name
            actor_color = color_name(board.turn)

        if move not in board.legal_moves:
            illegal_move = {"actor": actor, "move": move.uci(), "fen": board.fen()}
            termination = "illegal_move"
            break

        san = board.san(move)
        board.push(move)
        moves.append({"ply": len(moves) + 1, "actor": actor, "color": actor_color, "uci": move.uci(), "san": san, "fen": board.fen()})
        show_board(
            board,
            header=(
                f"{level_name} game {game_index} | "
                f"White={white_name} | Black={black_name} | "
                f"McChess={color_name(mcchess_color)} | "
                f"{actor} ({actor_color}) played {san} ({move.uci()})"
            ),
            last_move=move,
        )
        if SECONDS_BETWEEN_MOVES > 0:
            time.sleep(SECONDS_BETWEEN_MOVES)

    outcome = board.outcome(claim_draw=True)
    result = outcome.result() if outcome is not None else "1/2-1/2"
    if illegal_move is not None:
        result = "*"
    score = score_from_result(result, mcchess_color) if result != "*" else None
    winner = winner_from_result(result)
    completed_at = dt.datetime.now(dt.timezone.utc).isoformat()
    white_name = bot.name if mcchess_color == chess.WHITE else level_name
    black_name = level_name if mcchess_color == chess.WHITE else bot.name
    winner_name = white_name if winner == "white" else black_name if winner == "black" else winner
    show_board(
        board,
        header=(
            f"Done {level_name} game {game_index}: result={result}, winner={winner_name}, termination={termination} | "
            f"White={white_name} | Black={black_name} | McChess={color_name(mcchess_color)}"
        ),
    )

    return {
        "level": level_name,
        "game_index": game_index,
        "status": "failed" if illegal_move is not None else "completed",
        "result": result,
        "winner": winner,
        "winner_name": winner_name,
        "termination": termination,
        "mcchess_color": color_name(mcchess_color),
        "mcchess_score": score,
        "white": bot.name if mcchess_color == chess.WHITE else level_name,
        "black": level_name if mcchess_color == chess.WHITE else bot.name,
        "max_ply": MAX_PLY,
        "ply_count": len(moves),
        "final_fen": board.fen(),
        "moves": moves,
        "illegal_move": illegal_move,
        "stockfish_path": STOCKFISH_PATH,
        "stockfish_options": configured_options,
        "stockfish_limit": limit_config,
        "checkpoint": str(bot.checkpoint.metadata.path),
        "checkpoint_epoch": bot.checkpoint.metadata.epoch,
        "mcts_simulations": bot.config.simulations,
        "c_puct": bot.config.c_puct,
        "started_at": started_at,
        "completed_at": completed_at,
    }

In [ ]:
output_dir = project_root / "runs" / "external_stockfish"
output_dir.mkdir(parents=True, exist_ok=True)
all_games = []
global_game_index = 0
BOARD_DISPLAY = None
TABLE_DISPLAY = None
show_board(chess.Board(), header="Starting MCTS-200 vs Stockfish benchmark")
show_game_table(all_games)

engine = chess.engine.SimpleEngine.popen_uci(STOCKFISH_PATH)
try:
    display({"stockfish_id": engine.id, "available_options": sorted(engine.options)})
    for schedule_item in STOCKFISH_SCHEDULE:
        level_name = str(schedule_item["level"])
        games = int(schedule_item.get("games", 1))
        if games <= 0:
            raise ValueError(f"games must be positive for {level_name}")

        options = dict(schedule_item.get("options", {}))
        limit_config = dict(schedule_item.get("limit", {"time": 0.05}))
        configured_options = configure_stockfish(engine, options)

        level_games = []
        for level_game_index in range(games):
            mcchess_color = chess.WHITE if global_game_index % 2 == 0 else chess.BLACK
            game_record = play_stockfish_game(
                engine=engine,
                level_name=level_name,
                game_index=global_game_index,
                mcchess_color=mcchess_color,
                configured_options=configured_options,
                limit_config=limit_config,
            )
            all_games.append(game_record)
            level_games.append(game_record)
            game_path = output_dir / f"mcts_{MCTS_SIMULATIONS}_vs_stockfish_{level_name}_game_{global_game_index:03d}.json"
            game_path.write_text(json.dumps(game_record, indent=2, sort_keys=True) + "\n", encoding="utf-8")
            table_paths = write_game_table(output_dir, all_games)
            display({"saved_game": str(game_path), "result": game_record["result"], "winner": game_record["winner_name"], "mcchess_score": game_record["mcchess_score"], **table_paths})
            show_game_table(all_games)
            global_game_index += 1

        completed = [game for game in level_games if game["status"] == "completed"]
        score = sum(float(game["mcchess_score"]) for game in completed) / len(completed) if completed else 0.0
        display({"level": level_name, "completed_games": len(completed), "score": score})
finally:
    engine.quit()

completed_games = [game for game in all_games if game["status"] == "completed"]
elo_estimate = estimate_mcchess_elo(completed_games)
summary = {
    "status": "completed" if len(completed_games) == len(all_games) else "failed",
    "note": "External Stockfish reference benchmark only. Do not use as training data or engine labels.",
    "stockfish_path": STOCKFISH_PATH,
    "schedule": STOCKFISH_SCHEDULE,
    "checkpoint": str(bot.checkpoint.metadata.path),
    "checkpoint_epoch": bot.checkpoint.metadata.epoch,
    "mcts_simulations": bot.config.simulations,
    "c_puct": bot.config.c_puct,
    "games_requested": sum(int(item.get("games", 1)) for item in STOCKFISH_SCHEDULE),
    "games_completed": len(completed_games),
    "score": sum(float(game["mcchess_score"]) for game in completed_games) / len(completed_games) if completed_games else 0.0,
    "elo_estimate": elo_estimate,
    "games": all_games,
}
summary_path = output_dir / f"mcts_{MCTS_SIMULATIONS}_vs_stockfish_summary.json"
summary_path.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")
table_paths = write_game_table(output_dir, all_games)
display({"saved_summary": str(summary_path), "games_completed": summary["games_completed"], "score": summary["score"], **table_paths})
show_game_table(all_games)
display({"rough_stockfish_uci_elo_estimate": elo_estimate})